# Forward Simulation 1850–2014

SMB: MAR-corrected GISS monthly T/P  
Calving: Von Mises  
Basal melt: uniform floating ice melt (10 m/yr)  
Frontal melt: SMR annual means (Xu & Rignot 2012)

In [ ]:
import os
import sys
import numpy as np
from netCDF4 import Dataset
import xarray as xr
from scipy.interpolate import griddata

sys.path.append("/home/yanmeiti/issm/Functions/")
sys.path.append(os.getenv('ISSM_DIR') + '/bin')
sys.path.append(os.getenv('ISSM_DIR') + '/lib')
sys.path.append(os.getenv('ISSM_DIR') + '/share')
sys.path.append(os.getenv('ISSM_DIR') + '/share/proj')

if 'SMBd18opdd' in sys.modules:
    sys.modules.pop('SMBd18opdd', None)

my_function_path = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Functions/'
if my_function_path not in sys.path:
    sys.path.insert(0, my_function_path)

from model import *
from loadmodel import loadmodel
from InterpFromGridToMesh import InterpFromGridToMesh
from ll2xy import ll2xy
from xy2ll import xy2ll
from solve import solve
from export_netCDF import export_netCDF
from generic import generic
from cfl_step_v2 import cfl_step
from calvingvonmises import calvingvonmises
from ContourToMesh import ContourToMesh
from frontalforcings import frontalforcings


## 1. Load calving relaxation result

In [ ]:
print('Loading model...')
loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step2.2_relaxation_calving_v2.nc'
md = loadmodel(loadname)
print('Model loaded.')

# update geometry and initialization from last relaxation step
md.geometry.thickness       = md.results.TransientSolution[-1].Thickness.copy()
md.geometry.base            = md.results.TransientSolution[-1].Base.copy()
md.geometry.base            = np.maximum(md.geometry.base, md.geometry.bed)
md.geometry.surface         = md.geometry.base + md.geometry.thickness
md.initialization.vel       = md.results.TransientSolution[-1].Vel.copy()
md.initialization.vx        = md.results.TransientSolution[-1].Vx.copy()
md.initialization.vy        = md.results.TransientSolution[-1].Vy.copy()
md.initialization.pressure  = md.results.TransientSolution[-1].Pressure.copy()

# update ice levelset from last step
if hasattr(md.results.TransientSolution[-1], 'MaskIceLevelset'):
    md.mask.ice_levelset = md.results.TransientSolution[-1].MaskIceLevelset.copy()

print(f'Surface: {md.geometry.surface.min():.1f} - {md.geometry.surface.max():.1f} m')
print(f'Velocity max: {md.initialization.vel.max():.1f} m/yr')
print(f'StressbalanceSolution Vel max: {md.results.StressbalanceSolution.Vel.max():.1f} m/yr')


## 2. SMB: MAR climatology + GISS 1850-2014 (MAR-corrected)

In [ ]:
md.smb = SMBd18opdd()
md.smb.isd18opd = 1
md.smb.delta18o = np.array([[-40.0110], [0.0]])
md.smb.rlaps    = 6.0
md.smb.desfac   = 1
md.smb.rlapslgm = 5.5
md.smb.issetpddfac  = 1
md.smb.pddfac_snow  = 4.0 * np.ones(md.mesh.numberofvertices)
md.smb.pddfac_ice   = 8.0 * np.ones(md.mesh.numberofvertices)
md.smb.isprecipscaled      = 0
md.smb.istemperaturescaled = 0

# ── MAR 1981-2010 climatology ──
MAR_file = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Data/MAR/MAR_1981_2010_monthly_climatology_for_ISSM.nc'
ds_MAR  = xr.open_dataset(MAR_file)
tt_MAR  = ds_MAR['TT'].values
pr_MAR  = ds_MAR['PR'].values
sh_MAR  = ds_MAR['SH'].values
lat_MAR = ds_MAR['LAT'].values
lon_MAR = ds_MAR['LON'].values
ds_MAR.close()

[md.mesh.lat, md.mesh.long] = xy2ll(md.mesh.x, md.mesh.y, +1)
pts = np.column_stack((lat_MAR.ravel(), lon_MAR.ravel()))

md.smb.precipitations_presentday = np.full((md.mesh.numberofvertices, 12), np.nan)
md.smb.temperatures_presentday   = np.full((md.mesh.numberofvertices, 12), np.nan)
for i in range(12):
    md.smb.precipitations_presentday[:, i] = griddata(pts, pr_MAR[i].ravel(), (md.mesh.lat, md.mesh.long), method='nearest')
    md.smb.temperatures_presentday[:, i]   = griddata(pts, tt_MAR[i].ravel(), (md.mesh.lat, md.mesh.long), method='nearest')

MAR_dem    = griddata(pts, sh_MAR.ravel(), (md.mesh.lat, md.mesh.long), method='nearest')
md.smb.s0p = np.maximum(MAR_dem, 0)
md.smb.s0t = np.maximum(MAR_dem, 0)
print('MAR climatology done.')
print(f'  T: {md.smb.temperatures_presentday.min():.2f} - {md.smb.temperatures_presentday.max():.2f} K')
print(f'  P: {md.smb.precipitations_presentday.min():.4f} - {md.smb.precipitations_presentday.max():.4f} m/yr')


In [ ]:
# ── GISS 1850-2014 monthly T/P ──
tas_file = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/GISS_Data/ssp245/r1i1p1f2/historic/tas_historical_r1i1p1f2_185001-201412.nc'
pr_file  = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/GISS_Data/ssp245/r1i1p1f2/historic/pr_historical_r1i1p1f2_185001-201412.nc'

nc_tas   = Dataset(tas_file)
lat_giss = nc_tas.variables['lat'][:]
lon_giss = nc_tas.variables['lon'][:]
tas_raw  = nc_tas.variables['tas'][:]
nc_tas.close()
nc_pr    = Dataset(pr_file)
pr_raw   = nc_pr.variables['pr'][:]
nc_pr.close()

lon_giss = (lon_giss + 180) % 360 - 180
lat_idx  = np.where((lat_giss >= 50) & (lat_giss <= 90))[0]
lon_idx  = np.where((lon_giss >= -80) & (lon_giss <= -5))[0]
i_lat0, i_lat1 = lat_idx.min(), lat_idx.max() + 1
i_lon0, i_lon1 = lon_idx.min(), lon_idx.max() + 1

lat_sub   = lat_giss[i_lat0:i_lat1]
lon_sub   = lon_giss[i_lon0:i_lon1]
tas_sub   = tas_raw[:, i_lat0:i_lat1, i_lon0:i_lon1]
pr_sub    = pr_raw[:,  i_lat0:i_lat1, i_lon0:i_lon1]
pr_reunit = pr_sub * 3.154e7 * (1/1000) * (md.materials.rho_freshwater / md.materials.rho_ice)

num_vertices = md.mesh.numberofvertices
nt     = tas_sub.shape[0]
x_axis = lon_sub.astype(float)
y_axis = lat_sub.astype(float)

temp_series   = np.zeros((num_vertices, nt))
precip_series = np.zeros((num_vertices, nt))

print(f'Interpolating {nt} months to mesh...')
for k in range(nt):
    if k % 240 == 0:
        print(f'  month {k}/{nt}')
    temp_series[:, k]   = InterpFromGridToMesh(x_axis, y_axis, tas_sub[k],   md.mesh.long, md.mesh.lat, 0)
    precip_series[:, k] = InterpFromGridToMesh(x_axis, y_axis, pr_reunit[k], md.mesh.long, md.mesh.lat, 0)
print('Done.')

years_giss  = 1850 + np.arange(nt) // 12
months_giss = np.arange(nt) % 12
months_1idx = months_giss + 1
time_decimal = years_giss + (months_1idx - 0.5) / 12.0

ref_mask         = (years_giss >= 1981) & (years_giss <= 2010)
temp_ref         = temp_series[:, ref_mask].reshape(num_vertices, -1, 12)
precip_ref       = precip_series[:, ref_mask].reshape(num_vertices, -1, 12)
temp_giss_clim   = np.mean(temp_ref,   axis=1)
precip_giss_clim = np.mean(precip_ref, axis=1)
precip_giss_clim[precip_giss_clim <= 0] = 1e-6

tmp = np.zeros((num_vertices, nt))
pre = np.zeros((num_vertices, nt))
for k in range(nt):
    m = months_giss[k]
    dT      = temp_series[:, k] - temp_giss_clim[:, m]
    p_ratio = precip_series[:, k] / precip_giss_clim[:, m]
    tmp[:, k] = md.smb.temperatures_presentday[:, m] + dT
    pre[:, k] = md.smb.precipitations_presentday[:, m] * p_ratio
pre[pre <= 0] = 0.1

print(f'T range: {tmp.min():.2f} - {tmp.max():.2f} K')
print(f'P range: {pre.min():.4f} - {pre.max():.4f} m/yr')


In [ ]:
md.smb.temperatures_reconstructed   = np.zeros((num_vertices + 1, nt))
md.smb.precipitations_reconstructed = np.zeros((num_vertices + 1, nt))
md.smb.temperatures_reconstructed[:-1, :]   = tmp
md.smb.precipitations_reconstructed[:-1, :] = pre
md.smb.temperatures_reconstructed[-1, :]    = time_decimal
md.smb.precipitations_reconstructed[-1, :]  = time_decimal

delta18o = np.loadtxt('/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Data/delta18o.data')
md.smb.delta18o = delta18o

print(f'Reconstructed T/P shape: {md.smb.temperatures_reconstructed.shape}')
print(f'Time: {time_decimal[0]:.3f} - {time_decimal[-1]:.3f}')


In [ ]:
# visualize the reconstructed Tas/Pr

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np

tri = mtri.Triangulation(md.mesh.x, md.mesh.y, md.mesh.elements[:, :3] - 1)

def plot_tri_field(ax, values, title, cmap="viridis", vmin=None, vmax=None):
    tpc = ax.tripcolor(tri, values, shading="flat", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.set_aspect("equal")
    plt.colorbar(tpc, ax=ax, fraction=0.046, pad=0.04)

# fields to compare
t_fields = [tmp[:, 0], tmp[:, 6], tmp[:, -12], tmp[:, -6]]
p_fields = [pre[:, 0], pre[:, 6], pre[:, -12], pre[:, -6]]

tmin = np.nanmin([arr.min() for arr in t_fields])
tmax = np.nanmax([arr.max() for arr in t_fields])

pmin = np.nanmin([arr.min() for arr in p_fields])
pmax = np.nanmax([arr.max() for arr in p_fields])

fig, axes = plt.subplots(2, 4, figsize=(19, 10), constrained_layout=True)

# first row: temperature
plot_tri_field(axes[0, 0], tmp[:, 0],   "T 1850-01 (K)", cmap="turbo",   vmin=tmin, vmax=tmax)
plot_tri_field(axes[0, 1], tmp[:, 6],   "T 1850-07 (K)", cmap="turbo",   vmin=tmin, vmax=tmax)
plot_tri_field(axes[0, 2], tmp[:, -12], "T 2014-01 (K)", cmap="turbo",   vmin=tmin, vmax=tmax)
plot_tri_field(axes[0, 3], tmp[:, -6],  "T 2014-07 (K)", cmap="turbo",   vmin=tmin, vmax=tmax)

# second row: precipitation
plot_tri_field(axes[1, 0], pre[:, 0],   "P 1850-01 (m/yr)", cmap="viridis", vmin=pmin, vmax=pmax)
plot_tri_field(axes[1, 1], pre[:, 6],   "P 1850-07 (m/yr)", cmap="viridis", vmin=pmin, vmax=pmax)
plot_tri_field(axes[1, 2], pre[:, -12], "P 2014-01 (m/yr)", cmap="viridis", vmin=pmin, vmax=pmax)
plot_tri_field(axes[1, 3], pre[:, -6],  "P 2014-07 (m/yr)", cmap="viridis", vmin=pmin, vmax=pmax)

plt.show()


## 3. Frontal forcing: submarine melt rate (annual mean)

In [ ]:
print('Loading submarine melt rate...')
smr_file = '/home/yanmeiti/issm/TF/giss-forcing-pipeline/giss_data/r1i1p1f2/output/SMR_E2-1-G_r1i1p1f2_1850-2014.nc'
nc_smr   = Dataset(smr_file)
x_smr    = nc_smr.variables['x'][:]          # (1681,) m  ISMIP6 projection
y_smr    = nc_smr.variables['y'][:]          # (2881,) m
t_days   = nc_smr.variables['time'][:]       # days since 1850-01-01
smr_raw  = nc_smr.variables['submarine_melt'][:]  # (1969, 2881, 1681) m/yr
nc_smr.close()

# convert time: days → decimal year
t_dec_smr = 1850.0 + t_days / 365.25        # (1969,)

# annual averaging: group months by year → 165 annual means (1850-2014)
year_int     = t_dec_smr.astype(int)
uniq_years   = np.arange(1850, 2015)           # 165 years
nt_ann       = len(uniq_years)
time_smr_ann = uniq_years + 0.5               # mid-year decimal

nv = md.mesh.numberofvertices
smr_mesh_ann = np.zeros((nv, nt_ann))

print(f'Interpolating SMR ({nt_ann} annual steps) to mesh...')
for yi, yr in enumerate(uniq_years):
    if yi % 20 == 0:
        print(f'  year {yr}')
    mask_yr = (year_int == yr)
    if mask_yr.sum() == 0:
        continue
    field_ann = np.nanmean(smr_raw[mask_yr, :, :], axis=0)
    field_ann = np.where(np.isfinite(field_ann), field_ann, 0.0)
    field_ann = np.maximum(field_ann, 0.0)
    smr_mesh_ann[:, yi] = InterpFromGridToMesh(
        x_smr.astype(float), y_smr.astype(float),
        field_ann, md.mesh.x, md.mesh.y, 0)

smr_mesh_ann = np.clip(smr_mesh_ann, 0.0, 500.0)   # cap at 500 m/yr

md.frontalforcings.meltingrate = np.zeros((nv + 1, nt_ann))
md.frontalforcings.meltingrate[:-1, :] = smr_mesh_ann
md.frontalforcings.meltingrate[-1, :]  = time_smr_ann

print(f'SMR range: {smr_mesh_ann.min():.2f} - {smr_mesh_ann.max():.2f} m/yr')
print(f'SMR time: {time_smr_ann[0]:.1f} - {time_smr_ann[-1]:.1f}')


## 4. Basal forcing: simple uniform floating ice melt

In [ ]:
# Simple basal melt: uniform rate for floating ice, zero for grounded
# PICO not used — Greenland has limited floating ice and uncertain ocean forcing
# floatingice_melting_rate: typical values 10-30 m/yr for Greenland fjords
# Set to 10 m/yr as a conservative estimate; adjust based on sensitivity tests

md.basalforcings.floatingice_melting_rate = 10.0 * np.ones(md.mesh.numberofvertices)
md.basalforcings.groundedice_melting_rate = np.zeros(md.mesh.numberofvertices)

print('Basal forcings set.')
print(f'  Floating ice melt rate: {md.basalforcings.floatingice_melting_rate.mean():.1f} m/yr (uniform)')
print(f'  Grounded ice melt rate: 0 m/yr')


## 5. Transient settings + calving

In [ ]:
print('Setting up transient...')
time_step_cfl = cfl_step(md, md.initialization.vx, md.initialization.vy)
print(f'CFL time step: {time_step_cfl:.3f} yr')

md.timestepping.start_time = 1850.0
md.timestepping.final_time = 1855.0
md.timestepping.time_step  = 0.05

md.transient.isthermal     = 0
md.transient.ismovingfront = 1
md.inversion.iscontrol     = 0

md.transient.requested_outputs = [
    'default',
    'Thickness',
    'MaskIceLevelset',
    'MaskOceanLevelset',
    'CalvingCalvingrate',
    'CalvingMeltingrate',
    'SmbMassBalance',
    'SmbMelt',
    'TotalSmb',
    'TemperaturePDD',
    'IceVolume',
    'IceVolumeAboveFloatation',
    'GroundedArea',
    'FloatingArea',
    'IceVolumeScaled',
    'IceVolumeAboveFloatationScaled',
    'GroundedAreaScaled',
    'FloatingAreaScaled',
    'TotalSmbScaled',
]

md.settings.output_frequency = 20   # every 20 steps × 0.05 yr = 1 yr

# von Mises calving
md.levelset.spclevelset = np.full(md.mesh.numberofvertices, np.nan)
pos_spc = np.union1d(np.where(md.mesh.vertexonboundary > 0)[0],
                     np.where(md.mask.ice_levelset > 0)[0])
md.levelset.spclevelset[pos_spc] = md.mask.ice_levelset[pos_spc]
md.levelset.reinit_frequency  = 5
md.levelset.stabilization     = 1
md.thermal.reltol             = 0.01
md.thermal.penalty_lock       = 2
md.thermal.penalty_threshold  = 10

md.calving = calvingvonmises()
md.calving.stress_threshold_floatingice = 200e3   # Pa
md.calving.stress_threshold_groundedice = 1e6
md.calving.min_thickness                = 2
md.levelset.kill_icebergs               = 1


## 6. Solve

In [ ]:
md.miscellaneous.name = 'GrIS_forward_1850-2014_v2'
md.cluster = generic('name', 'localhost', 'np', 8)
md.verbose  = verbose('solution', True, 'module', False, 'convergence', False)
md.settings.solver_residue_threshold = 1e-4
md.stressbalance.maxiter = 100

print('Starting forward simulation (1901-2014)...')
md = solve(md, 'Transient')
print('Forward simulation completed.')


## 7. Save

In [ ]:
savename = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step3_forward_1850-2014.nc'
export_netCDF(md, savename)
print(f'Saved to {savename}')


## 8. Quick diagnostics

In [ ]:
n   = len(md.results.TransientSolution)
print(f'Total output steps: {n}')

years_out = [md.results.TransientSolution[i].time for i in range(n)]
vol       = [md.results.TransientSolution[i].IceVolume[0]              for i in range(n)]
smb_tot   = [md.results.TransientSolution[i].TotalSmb[0]               for i in range(n)]
ivaf      = [md.results.TransientSolution[i].IceVolumeAboveFloatation[0] for i in range(n)]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(years_out, vol,     'b-o', ms=3)
axes[0].set_ylabel('Ice Volume (m\u00b3)')
axes[0].set_title('Ice Volume Evolution')

axes[1].plot(years_out, smb_tot, 'g-o', ms=3)
axes[1].set_ylabel('Total SMB (m\u00b3/yr)')
axes[1].set_title('Total SMB')

axes[2].plot(years_out, ivaf,    'r-o', ms=3)
axes[2].set_ylabel('IVAF (m\u00b3)')
axes[2].set_xlabel('Year')
axes[2].set_title('Ice Volume Above Floatation')

plt.tight_layout()
plt.show()

V_end = md.results.TransientSolution[-1].Vel
H_end = md.results.TransientSolution[-1].Thickness
print(f'\nFinal velocity: max={np.nanmax(V_end):.1f}  p99={np.nanpercentile(V_end,99):.1f} m/yr')
print(f'Final thickness: max={np.nanmax(H_end):.1f}  min={np.nanmin(H_end):.1f} m')


In [ ]:
import numpy as np

rho_ice  = md.materials.rho_ice
Gt       = 1e-12
elements = md.mesh.elements.astype(int) - 1
x, y     = md.mesh.x, md.mesh.y

n = len(md.results.TransientSolution)
years, fw_frontal, fw_calving, fw_smb = [], [], [], []

for k in range(n):
    step     = md.results.TransientSolution[k]
    yr       = step.time
    ice_flag = (np.array(step.MaskIceLevelset).ravel() < 0).astype(int)
    H        = np.array(step.Thickness).ravel()
    smr      = np.array(step.CalvingMeltingrate).ravel()
    cr       = np.array(step.CalvingCalvingrate).ravel()

    # ── calving front edges: interior edges where the levelset crosses zero ──
    front_edge_set = set()
    for tri in elements:
        for k2 in range(3):
            i, j = tri[k2], tri[(k2+1)%3]
            if ice_flag[i] + ice_flag[j] == 1:
                front_edge_set.add(tuple(sorted([i, j])))

    # Edge-length weight for each front vertex
    vertex_w = np.zeros(md.mesh.numberofvertices)
    for (i, j) in front_edge_set:
        L = np.sqrt((x[i]-x[j])**2 + (y[i]-y[j])**2)
        vertex_w[i] += L / 2
        vertex_w[j] += L / 2
    vertex_w[ice_flag == 0] = 0

    # Direct integration: frontal melt and calving (Gt/yr, positive = freshwater into ocean)
    q_fm   = float(np.sum(smr * H * vertex_w)) * rho_ice * Gt
    q_calv = float(np.sum(cr  * H * vertex_w)) * rho_ice * Gt
    q_smb  = float(step.TotalSmb[0])   # Gt/yr, for reference only

    years.append(yr)
    fw_frontal.append(q_fm)
    fw_calving.append(q_calv)
    fw_smb.append(q_smb)

print('Year | FW_frontal(Gt/yr) | FW_calving(Gt/yr) | SMB(Gt/yr)')
for i in range(n):
    print(f'{years[i]:.1f} | {fw_frontal[i]:8.2f} | {fw_calving[i]:8.2f} | {fw_smb[i]:8.2f}')
